In [ ]:
!cp -r /kaggle/input/datasets/lhuyton/sam3-source/sam3 /kaggle/working

In [ ]:
# Di chuyển vào thư mục code
%cd /kaggle/working/sam3

# Cài đặt offline các dependencies từ Dataset
!pip install --no-index --find-links=/kaggle/input/datasets/lhuyton/sam3-wheels/sam3_wheels .[train] --no-build-isolation

In [ ]:
!cp /kaggle/input/datasets/lhuyton/roboflow-config/roboflow_v100_full_ft_100_images.yaml /kaggle/working/sam3/sam3/train/configs/roboflow_v100/roboflow_v100_full_ft_100_images.yaml

In [ ]:
!cp /kaggle/input/datasets/lhuyton/builder/model_builder.py /kaggle/working/sam3/sam3/model_builder.py

In [ ]:
!cp /kaggle/input/datasets/lhuyton/fusedpy/fused.py /kaggle/working/sam3/sam3/perflib/fused.py

In [ ]:
!gzip -f /kaggle/working/sam3/sam3/assets/bpe_simple_vocab_16e6.txt

In [ ]:
import re
import os

# ==========================================
# 1. SỬA ĐỔI FILE CẤU HÌNH YAML
# ==========================================
yaml_path = '/kaggle/working/sam3/sam3/train/configs/roboflow_v100/roboflow_v100_full_ft_100_images.yaml'

with open(yaml_path, 'r') as f:
    yaml_content = f.read()

# Tối ưu VRAM & CPU
yaml_content = re.sub(r'num_train_workers:\s*\d+', 'num_train_workers: 32', yaml_content)
yaml_content = re.sub(r'num_val_workers:\s*\d+', 'num_val_workers: 8', yaml_content)
yaml_content = re.sub(r'val_batch_size:\s*\d+', 'val_batch_size: 16', yaml_content)
yaml_content = re.sub(r'train_batch_size:\s*\d+', 'train_batch_size: 16', yaml_content)

# Mở khóa toàn bộ dữ liệu & Chỉnh quy mô
yaml_content = re.sub(r'num_images:\s*\d+', 'num_images: null', yaml_content)
yaml_content = re.sub(r'target_epoch_size:\s*\d+', 'target_epoch_size: 10000', yaml_content)
yaml_content = re.sub(r'max_epochs:\s*\d+', 'max_epochs: 50', yaml_content) # Train từ đầu cần nhiều epoch

# Chống tràn ổ cứng Kaggle
yaml_content = re.sub(r'save_freq:\s*\d+', 'save_freq: 0', yaml_content)
yaml_content = re.sub(r'skip_saving_ckpts:\s*(true|True|false|False)', 'skip_saving_ckpts: false', yaml_content)

yaml_content = re.sub(r'lr_scale:\s*[0-9eE.-]+', 'lr_scale: 0.1', yaml_content)
yaml_content = re.sub(r'task_index: 0', 'task_index: 3', yaml_content)
with open(yaml_path, 'w') as f:
    f.write(yaml_content)


In [ ]:
# Trở về thư mục gốc của project (nơi chứa thư mục sam3 bên trong)
%cd /kaggle/working/sam3

# Chạy file train.py kèm theo PYTHONPATH để trỏ đúng thư mục hiện tại
!PYTHONPATH=$(pwd) python sam3/train/train.py -c configs/roboflow_v100/roboflow_v100_full_ft_100_images --use-cluster 0 --num-gpus 1

In [ ]:
!mv /kaggle/working/experiment_logs/checkpoints/checkpoint.pt /kaggle/working/

In [ ]:
!rm /kaggle/working/checkpoint.pt

In [ ]:
%cd /kaggle/working/experiment_logs/checkpoints
!ls